In [1]:
import os
import keras
from keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope
from hgq.layers import QDense, QSoftmax
from hgq.utils.sugar import FreeEBOPs, PBar



2026-04-23 18:36:09.665227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776962169.683929  709615 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776962169.689175  709615 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-23 18:36:09.710167: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets, can also be binarized instead
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
#X_train, X_test = (X_train > 127).astype(np.float32), (X_test > 127).astype(np.float32)

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]

# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)
y_val = keras.utils.to_categorical(y_val, 10)

dataset_train = Dataset(X_train, y_train, batch_size=2048, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=2048, device='gpu:0')

X_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1776962173.313281  709615 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [3]:
import keras
from hgq.layers import QDense, QConv2D
from hgq.config import LayerConfigScope, QuantizerConfigScope

with (
      QuantizerConfigScope(place='all', default_q_type='kbi', f0=5, i0=2, overflow_mode='SAT_SYM', trainable=True),
      #QuantizerConfigScope(place='bias', default_q_type='kif', f0=5, i0=2, overflow_mode='WRAP', trainable=True),
      QuantizerConfigScope(place='datalane', default_q_type='kif', f0=5, i0=2, heterogeneous_axis=None, homogeneous_axis=(0, 1)),
      LayerConfigScope(enable_ebops=True, beta0=1e-6),
   ):
      inputs = keras.Input(shape=input_shape)
      x = keras.layers.Flatten(input_shape=[28, 28])(inputs)
      x = QDense(32, activation='relu')(x)
      x = QDense(32, activation='relu')(x)
      outputs = QDense(10)(x)
      model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-3),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
    jit_compile=True,
    steps_per_execution=10
)

/home/slopin/hls4ml_proj/venv_HGQ/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense (QDense)                │ (None, 32)             │       100,485 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_1 (QDense)              │ (None, 32)             │         4,229 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_2 (QDense)              │ (None, 10)             │         1,325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,039 (336.55 KB)

 Trainable params: 79,524 (310.64 KB)

 Non-trainable params: 26,515 (25.91 KB)

In [5]:
import keras
import numpy as np
from math import cos, pi
from hgq.utils.sugar import BetaScheduler, Dataset, FreeEBOPs, ParetoFront, PBar, PieceWiseSchedule, EarlyStoppingWithEbopsThres
from keras.callbacks import LearningRateScheduler, CSVLogger

OUTPUT_PATH = '/home/slopin/DAT255-project/Modeller/model_HGQ/model_outputs_static_training_MLP'

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)
    print(f"Created new folder: {OUTPUT_PATH}")


pbar = PBar(
        'loss: {loss:.2f}/{val_loss:.2f} - acc: {accuracy:.4f}/{val_accuracy:.4f}'
    )
ebops = FreeEBOPs()
pareto = ParetoFront(
        OUTPUT_PATH,
        ['val_accuracy', 'ebops'],
        [1, -1],
        fname_format='epoch={epoch}-val_acc={val_accuracy:.3f}-ebops={ebops}-val_loss={val_loss:.3f}.keras',
    )
estop = EarlyStoppingWithEbopsThres(ebops_threshold=3000, monitor='val_accuracy', patience=30, restore_best_weights=True)

csv_logger = CSVLogger('/home/slopin/DAT255-project/Modeller/model_HGQ/MNIST_MLP_HGQ_StaticTraining_log.csv', append=True, separator=';')

In [6]:
callbacks = [ebops, pbar, pareto, estop, csv_logger]

In [7]:
model.fit(dataset_train, epochs=8000, validation_data=dataset_val, callbacks=callbacks, verbose=0)

  0%|          | 0/8000 [00:00<?, ?epoch/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1776962190.307894  709717 service.cc:148] XLA service 0x748ed4017e30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776962190.308092  709717 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-04-23 18:36:30.466731: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776962190.957090  709717 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-04-23 18:36:31.589752: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1619', 48 bytes spill stores, 48 bytes spill loads

2026-04-23 18:36:31.901705: I external/local_xla/xla/stre

KeyboardInterrupt: 